# Chapter 4: Debugging Hallucinations with Math
## A Deep Dive into Multi-Layer RAG Evaluation, Claim Grounding, and Policy Gates

This notebook demonstrates how a RAG system moves from an answer that merely *sounds* plausible to an answer that is measured across **five distinct mathematical layers**, traced in Opik Cloud, and deterministically gated.

In [ ]:
# Setup Python path and verify imports from ch04_eval
import os
import sys

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath(".."))

from ch04_eval.config import get_settings
from ch04_eval.generation import OllamaGenerator
from ch04_eval.grounding import GroundingJudge
from ch04_eval.ingest import load_corpus
from ch04_eval.policy_gate import PolicyGate, PolicyThresholds
from ch04_eval.retrieval import BM25Retriever, compute_deterministic_metrics
from ch04_eval.schemas import RiskTier

settings = get_settings()
print("Loaded configuration. Corpus path:", settings.corpus_path)

### Layer 1: Ingestion & Deterministic Retrieval Ranking
We parse the multi-document markdown policy manual into structured chunks with document IDs and calculate classical retrieval metrics ($Recall@K$, $Precision@K$, $MRR$).

In [ ]:
# Ingest the policy manual
chunks = load_corpus(os.path.join("..", settings.corpus_path))
print(f"Parsed {len(chunks)} discrete chunks across policy documents.")

# Build BM25 Index
retriever = BM25Retriever(chunks)

# Test query 1: Core collaboration hours
query = "What are the core collaboration hours for full-time remote employees?"
retrieval_result = retriever.retrieve(query, top_k=5)

print(
    f"Top retrieved chunk: [{retrieval_result.retrieved_chunks[0].chunk_id}] (Score: {retrieval_result.retrieved_chunks[0].retrieval_score})"
)
metrics = compute_deterministic_metrics(retrieval_result, ["HR-2026-01"], k=5)
print(f"Recall@5: {metrics.recall_at_k} | MRR: {metrics.mrr}")

### Layer 2: Grounded Generation with Ollama Cloud
The generator produces responses constrained strictly to the retrieved context, citing exact source chunk tags (`[doc_id#chunk_num]`).

In [ ]:
generator = OllamaGenerator(settings=settings)
gen_result = generator.generate(query, retrieval_result.retrieved_chunks)

print("--- Generated Answer ---")
print(gen_result.answer)
print("\nExtracted Citations:", gen_result.citations)

### Layer 3 & 4: Claim-Level Grounding & Evidence Sufficiency Judge
We break down the answer into atomic claims and verify them against the evidence chunks, computing $\text{claim\_grounding\_rate}$.

In [ ]:
judge = GroundingJudge(settings=settings)

# Evaluate atomic claim grounding
grounding_result = judge.evaluate_grounding(
    query, gen_result.answer, retrieval_result.retrieved_chunks
)
print(f"Claim Grounding Rate: {grounding_result.claim_grounding_rate:.2f}")
for c in grounding_result.claims:
    print(f"  - [{c.verdict.value}] Claim: {c.claim}")

# Evaluate evidence sufficiency
sufficiency_result = judge.evaluate_sufficiency(query, retrieval_result.retrieved_chunks)
print(f"\nEvidence Sufficiency Class: {sufficiency_result.sufficiency_class.value}")
print(f"Judge Rationale: {sufficiency_result.rationale}")

### Layer 5: Deterministic Policy Gating
The policy gate evaluates signals deterministically and decides: `ANSWER`, `QUALIFIED_ANSWER`, `ABSTAIN`, `BLOCK`, or `HUMAN_REVIEW`.

In [ ]:
policy_gate = PolicyGate(thresholds=PolicyThresholds())
decision_result = policy_gate.evaluate(
    risk_tier=RiskTier.LOW,
    deterministic_metrics=metrics,
    ragas_metrics=None,  # Handled with defaults
    grounding_result=grounding_result,
    sufficiency_result=sufficiency_result,
)

print("Final Policy Decision:", decision_result.decision.value)
print("Reason:", decision_result.decision_reason)

### Edge Case Demonstration: Out-of-Corpus Query (Safe Abstention)
Now let's trace an out-of-corpus query (Paid Parental Leave duration) to verify how the pipeline abstains rather than inventing plausible benefits.

In [ ]:
out_of_corpus_query = "How many weeks of paid parental leave are employees entitled to receive?"
res_edge = retriever.retrieve(out_of_corpus_query, top_k=5)
gen_edge = generator.generate(out_of_corpus_query, res_edge.retrieved_chunks)
suff_edge = judge.evaluate_sufficiency(out_of_corpus_query, res_edge.retrieved_chunks)

print("--- Edge Case Generated Answer ---")
print(gen_edge.answer)
print("\nSufficiency Classification:", suff_edge.sufficiency_class.value)

gate_edge = policy_gate.evaluate(
    risk_tier=RiskTier.MEDIUM,
    deterministic_metrics=compute_deterministic_metrics(res_edge, [], k=5),
    ragas_metrics=None,
    grounding_result=judge.evaluate_grounding(
        out_of_corpus_query, gen_edge.answer, res_edge.retrieved_chunks
    ),
    sufficiency_result=suff_edge,
)

print("\nPolicy Gate Decision:", gate_edge.decision.value)
print("Reason:", gate_edge.decision_reason)